# Network Addressing and Routing

So far, you have used the shell to work with files and programs in the environment where your commands run. Working with a Duckiedrone adds another requirement: your development computer must exchange data with a separate computer on the robot.

For example, opening a Duckiedrone’s dashboard requires your computer to know where to send its request. Being connected to Wi-Fi is only part of the answer: which address belongs to the Duckiedrone, and how should data travel there?

This notebook introduces the information computers use to make those decisions. You will inspect network interfaces and addresses, identify a local subnet, and read a routing table to distinguish destinations reached locally from those reached through a gateway.

The worked examples come from a physical Duckiedrone named `amelia`. Its addresses describe the network used when the output was recorded; your setup will probably use different values.

> **Where to run the commands**
>
> These are Linux commands. On an Ubuntu base station, run them in its own terminal to inspect that computer.
>
> In a Duckietown Workspace, the same commands describe the development environment’s network configuration. Its interfaces and gateway may differ from those of the physical computer running it.
>
> You can complete the activities using Amelia’s recorded output below. Connecting to a Duckiedrone through SSH is covered later in this LX.

## Local networks

Suppose you want to open a Duckiedrone’s dashboard from your base station—the development computer you use to work with the robot. The two computers need a connection over which they can exchange data.

A common arrangement is for both to join the same local Wi-Fi network:

```text
Base station                         Duckiedrone
     |                                    |
     +------------ Wi-Fi -----------------+
                       |
             Wireless access point
                       |
                Local network
                       |
                     Router
                       |
                Other networks
             (including the internet)
```

This is an example arrangement, not a required wiring diagram. The access point and router may be parts of the same physical device.

A **wireless access point** connects Wi-Fi devices to a local network. If you connect the base station with an Ethernet cable instead, a **switch** can connect it to other devices on that network. A **router** connects different networks and forwards data between them.

A home Wi-Fi router commonly combines these roles in one box. The distinction matters because exchanging data with a nearby Duckiedrone and reaching a website on the internet do not necessarily use the same path.

Computers send network data in units called **packets**. An Internet Protocol (**IP**) packet includes source and destination IP addresses, which identify where it comes from and where it is intended to go.

For communication within the local network, packets may pass through an access point or switch without being routed to another network. Consequently, a local Duckiedrone connection can still work when the network’s internet connection is unavailable.

**Think it through:** if the internet connection in the diagram fails, but the local network keeps working, does that failure alone prevent the base station from communicating with the Duckiedrone?

<details>
<summary>Show explanation</summary>

No. Both computers are on the local network, so their communication does not require the internet connection. Whether the dashboard actually works also depends on the Duckiedrone and its software; we will investigate those additional requirements later.

</details>

## Interfaces and identifiers

Before choosing where to send a packet, a computer needs a connection through which to send it.

A **network interface** is such a connection. A computer might have an Ethernet interface for a cable, a Wi-Fi interface for wireless communication, and additional virtual interfaces created by software.

First, identify the computer where the command runs:

```bash
hostname
```

On our example Duckiedrone:

```text
amelia
```

This is its **hostname**, a human-readable name. Running `hostname` on your base station or inside your Workspace reports the name of that environment instead. The command also works in a macOS terminal.

### Find the interface with an address

On Linux, run:

```bash
ip -brief address
```

`-brief` requests a compact display. On `amelia`, the output is:

```text
lo               UNKNOWN        127.0.0.1/8 ::1/128
eth0             DOWN
wlan0            UP             192.168.1.201/24 metric 600
docker0          DOWN           172.17.0.1/16
```

Read each row as an interface name, its reported operational state, and any assigned IP addresses:

| Interface | What this capture tells us |
| --- | --- |
| `lo` | The loopback interface, used for communication within `amelia` itself. |
| `eth0` | The Ethernet interface is down and has no IP address shown; it may be disabled or lack an active link. |
| `wlan0` | The Wi-Fi interface is up and has IPv4 address `192.168.1.201`. |
| `docker0` | A virtual interface created by Docker. It has an address even though its operational state is down. |

For the physical network connection in this example, focus on `wlan0`. The `/24` beside its address describes the subnet; we will interpret it in the next section. The `metric 600` value concerns route preference, which we will revisit when reading routes.

An `UP` interface is useful evidence, but it does not establish that a particular remote computer or dashboard is reachable. Conversely, the loopback interface’s `UNKNOWN` state is normal here and does not mean it is broken.

Interface names vary between computers. Your Wi-Fi interface could be named differently from `wlan0`.

### Distinguish an IP address from a MAC address

The Wi-Fi interface also has a **Media Access Control (MAC) address**, used to identify it on its local network connection. A hardware interface usually has a factory-assigned MAC address, but software and Wi-Fi privacy features can change the address that the interface uses.

To inspect link information, run:

```bash
ip -brief link
```

The relevant row from `amelia`’s output is:

```text
wlan0            UP             dc:a6:32:31:43:ad <BROADCAST,MULTICAST,UP,LOWER_UP>
```

The values after the MAC address are interface flags. For now, use the two command outputs to compare the identifiers for the same interface:

| Identifier | amelia’s Wi-Fi interface |
| --- | --- |
| Interface name | `wlan0` |
| MAC address | `dc:a6:32:31:43:ad` |
| IPv4 address | `192.168.1.201` |

These identifiers serve different purposes. `wlan0` is the name Linux uses for the interface. Its MAC address identifies it on the local connection. Its IP address is used to address packets, including packets that may need to travel across routers.

For example, a network administrator might ask for the Duckiedrone’s Wi-Fi MAC address when registering it on a lab network so it can be authorized under the network’s access policy. A program contacting the Duckiedrone normally uses its hostname or IP address instead.

### Recognize addresses that stay inside the computer

Amelia’s `lo` row contains two loopback addresses:

```text
127.0.0.1/8 ::1/128
```

`127.0.0.1` is the familiar IPv4 loopback address; `::1` is the IPv6 equivalent. Traffic sent to these addresses stays within the current network environment.

## IPv4 subnets and gateways

Knowing Amelia’s address still leaves a question: which destinations can it reach on the local network, and which require a router?

An **IPv4 address** contains four decimal groups, each between `0` and `255`. The **subnet prefix** identifies the network portion of the address.

Consider Amelia’s Wi-Fi address:

```text
192.168.1.201/24
```

Each decimal group represents 8 bits. The `/24` says that the first 24 bits (the first three groups in this example) identify the network:

```text
192 . 168 . 1 . 201
└───────────┘   └─┘
 network part   host part
   24 bits       8 bits
```

Here, the subnet is `192.168.1.0/24`. The host portion distinguishes addresses within it. You may also see `/24` expressed as the **subnet mask** `255.255.255.0`.

Suppose a base station on the same local network has address `192.168.1.42/24`. 

```text
Base station                          Amelia
192.168.1.42/24                       192.168.1.201/24
       |                                    |
       +--------- 192.168.1.0/24 ------------+
                    Local subnet
```

Both addresses are on the same local subnet, so each host can select a direct route to the other without using a gateway; successful communication still depends on the link and access policy. Now consider another device with destination at, e.g., `192.168.2.42`. Its address is not in this Wi-Fi subnet, so both the base station and `amelia` would need a route beyond that local connection.

A **gateway** is a router used as the next step toward a destination. The **default gateway** handles destinations for which the computer has no more specific route.

| Destination from `amelia` | Relationship to `192.168.1.0/24` |
| --- | --- |
| `192.168.1.42` | Inside the Wi-Fi subnet |
| `192.168.1.77` | Inside the Wi-Fi subnet |
| `192.168.2.42` | Outside the Wi-Fi subnet |

Comparing the first three groups works here because the prefix is `/24`. Other prefix lengths divide the address differently. The subnet explains the local address range, but to see where Linux actually sends packets, we need its _routing table_.

## DNS

Numeric addresses are useful for understanding a network, but remembering an address for every robot would become inconvenient.

A **hostname lookup**, also called **name resolution**, finds IP addresses associated with a name. The **Domain Name System (DNS)** provides this mapping for names such as `duckietown.com`. Similarly, every Duckiedrone is given a name when first initialized, and through it the robot can be contacted on the local network, for example as `amelia.local`. If that name resolves (i.e., "translates") to Amelia’s recorded Wi-Fi address, the relationship would be:

```text
amelia.local
     |
     | name lookup
     v
192.168.1.201
     |
     | route selection
     v
outgoing network interface
```

Names ending in `.local` normally use **multicast DNS (mDNS)**, a local name-resolution mechanism. [Notebook 17 Network names and service discovery](./17-network-names-and-service-discovery.ipynb) explains how these names are resolved and how Duckietown device discovery uses the local network.

## Routing tables

The **routing table** contains rules that associate destination addresses with an outgoing interface and, when needed, a gateway. This information helps answer, for example, how does Linux choose between `wlan0`, `eth0`, and `docker0` if `amelia` needs to send data to `192.168.1.42` (the base station)?

In your terminal, run:

```bash
ip route
```

You should get something like this: 

```text
default via 192.168.1.1 dev wlan0 proto dhcp src 192.168.1.201 metric 600
172.17.0.0/16 dev docker0 proto kernel scope link src 172.17.0.1 linkdown
192.168.1.0/24 dev wlan0 proto kernel scope link src 192.168.1.201 metric 600
192.168.1.1 dev wlan0 proto dhcp scope link src 192.168.1.201 metric 600
```

Start with two entries: the route for the Wi-Fi subnet and the default route.

### Read the local route

```text
192.168.1.0/24 dev wlan0 proto kernel scope link src 192.168.1.201 metric 600
```

This entry describes destinations in `192.168.1.0/24`:

| Field | Meaning here |
| --- | --- |
| `192.168.1.0/24` | Destination subnet |
| `dev wlan0` | Send through the Wi-Fi interface |
| `scope link` | Destinations are on the directly connected network |
| `src 192.168.1.201` | Preferred source address for packets Amelia originates using this route |

There is no `via` gateway in this entry. For the example destination `192.168.1.42`, Amelia sends locally through `wlan0`.

“Locally” does not mean the Wi-Fi devices bypass the access point. It means the destination can be reached without routing the packet into another IP network.

### Read the default route

```text
default via 192.168.1.1 dev wlan0 proto dhcp src 192.168.1.201 metric 600
```

This says: for destinations without a more specific route, send through `wlan0` to the gateway at `192.168.1.1`.

For an illustrative destination at `192.168.2.42`, the first step would therefore be:

```text
Amelia                  Default gateway              Destination
192.168.1.201  --------> 192.168.1.1  ------ ? ------> 192.168.2.42
```

While the routing table tells us which gateway `amelia` would use; it does not tell us whether that gateway has a working onward path to the destination, hence the `?`.

### Understand which entry wins

For ordinary routing in this table, Linux uses the **most specific matching destination prefix**. A route for one subnet is more specific than the default route.

Consequently, the default route does not take precedence only because it appears first in the output.

| Example destination | Matching route selected from Amelia’s table | First step |
| --- | --- | --- |
| `192.168.1.42` | `192.168.1.0/24` | Directly over `wlan0` |
| `192.168.2.42` | `default` | Through `192.168.1.1` over `wlan0` |
| `192.168.1.1` | The entry specifically for `192.168.1.1` | Directly over `wlan0` to the gateway itself |

The remaining fields explain how the routes were installed or preferred. `proto kernel` marks a kernel-installed route; `proto dhcp` marks one supplied through automatic network configuration. A `metric` helps choose between otherwise comparable routes; lower values are preferred. These fields are documented in the [`ip route` manual](https://man7.org/linux/man-pages/man8/ip-route.8.html).

The `172.17.0.0/16` entry belongs to Docker’s virtual network. Its `linkdown` flag corresponds to the down state we saw for `docker0`. We do not need that route to explain Amelia’s Wi-Fi connection.

**Your turn:** suppose the default route disappeared, while the Wi-Fi interface and its local subnet route remained unchanged. Would Amelia still have a route to the example base station at `192.168.1.42`?

<details>
<summary>Show answer</summary>

Yes. The `192.168.1.0/24` route still covers that destination. Losing the default route does not remove the local route.

Amelia would, however, lack a route for destinations that previously depended on the default gateway.

</details>

A routing table describes forwarding decisions. It does not test whether another device responds. Notebook 18 combines route inspection with connection tests.

## IPv6 alongside IPv4

So far, we have followed IPv4 addresses. But Amelia’s interface output also contained `::1`, and a hostname lookup on another system might return an address containing letters and colons.

These are **IPv6 addresses**. IPv6 uses 128-bit addresses, compared with IPv4’s 32 bits, providing a much larger address space (to avoid internet addresses literally finishing, as the historical the internet grew bigger and bigger).

IPv6 writes addresses as hexadecimal groups separated by colons. A double colon, `::`, can replace one consecutive run of zero groups. For example, the documentation address:

```text
2001:db8:0:0:0:0:0:42
```

can be shortened to:

```text
2001:db8::42
```

Two forms are especially useful to recognize:

| Address | Meaning |
| --- | --- |
| `::1` | IPv6 loopback: the current network environment |
| An address beginning `fe80::` | A common form of IPv6 link-local address: usable on the local link and not forwarded by routers |

The [IPv6 addressing specification](https://www.rfc-editor.org/rfc/rfc4291.html) defines these address forms.

A computer can use IPv4 and IPv6 together, often called **dual stack**. Having IPv4 connectivity does not establish that IPv6 connectivity is available as well.

Look again at Amelia’s capture:

```text
lo               UNKNOWN        127.0.0.1/8 ::1/128
wlan0            UP             192.168.1.201/24 metric 600
```

Amelia has IPv6 loopback, but no IPv6 for Wi-Fi over `wlan0`, which is perfectly possible.

**Your turn:** if another computer shows both an IPv4 address and a `fe80::` address on its Wi-Fi interface, does that prove it can reach the internet over IPv6?

<details>
<summary>Show answer</summary>

No. The `fe80::` address is link-local and cannot be forwarded across a router. It does not establish an IPv6 internet connection.

</details>

## Further reading

For more detail on inspecting assigned addresses, consult the Linux [`ip address` manual](https://man7.org/linux/man-pages/man8/ip-address.8.html).

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()
